<a target="_blank" href="https://colab.research.google.com/github/AI4Finance-Foundation/FinRL-Tutorials/blob/master/2-Advance/FinRL_Ensemble_StockTrading_ICAIF_2020.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Deep Reinforcement Learning for Stock Trading from Scratch: Multiple Stock Trading Using Ensemble Strategy

Tutorials to use OpenAI DRL to trade multiple stocks using ensemble strategy in one Jupyter Notebook | Presented at ICAIF 2020

* This notebook is the reimplementation of our paper: Deep Reinforcement Learning for Automated Stock Trading: An Ensemble Strategy, using FinRL.
* Check out medium blog for detailed explanations: https://medium.com/@ai4finance/deep-reinforcement-learning-for-automated-stock-trading-f1dad0126a02
* Please report any issues to our Github: https://github.com/AI4Finance-LLC/FinRL-Library/issues
* **Pytorch Version**



# Content

* [1. Problem Definition](#0)
* [2. Getting Started - Load Python packages](#1)
    * [2.1. Install Packages](#1.1)    
    * [2.2. Check Additional Packages](#1.2)
    * [2.3. Import Packages](#1.3)
    * [2.4. Create Folders](#1.4)
* [3. Download Data](#2)
* [4. Preprocess Data](#3)        
    * [4.1. Technical Indicators](#3.1)
    * [4.2. Perform Feature Engineering](#3.2)
* [5.Build Environment](#4)  
    * [5.1. Training & Trade Data Split](#4.1)
    * [5.2. User-defined Environment](#4.2)   
    * [5.3. Initialize Environment](#4.3)    
* [6.Implement DRL Algorithms](#5)  
* [7.Backtesting Performance](#6)  
    * [7.1. BackTestStats](#6.1)
    * [7.2. BackTestPlot](#6.2)   
    * [7.3. Baseline Stats](#6.3)   
    * [7.3. Compare to Stock Market Index](#6.4)             

<a id='0'></a>
# Part 1. Problem Definition

This problem is to design an automated trading solution for single stock trading. We model the stock trading process as a Markov Decision Process (MDP). We then formulate our trading goal as a maximization problem.

The algorithm is trained using Deep Reinforcement Learning (DRL) algorithms and the components of the reinforcement learning environment are:


* Action: The action space describes the allowed actions that the agent interacts with the
environment. Normally, a ∈ A includes three actions: a ∈ {−1, 0, 1}, where −1, 0, 1 represent
selling, holding, and buying one stock. Also, an action can be carried upon multiple shares. We use
an action space {−k, ..., −1, 0, 1, ..., k}, where k denotes the number of shares. For example, "Buy
10 shares of AAPL" or "Sell 10 shares of AAPL" are 10 or −10, respectively

* Reward function: r(s, a, s′) is the incentive mechanism for an agent to learn a better action. The change of the portfolio value when action a is taken at state s and arriving at new state s',  i.e., r(s, a, s′) = v′ − v, where v′ and v represent the portfolio
values at state s′ and s, respectively

* State: The state space describes the observations that the agent receives from the environment. Just as a human trader needs to analyze various information before executing a trade, so
our trading agent observes many different features to better learn in an interactive environment.

* Environment: Dow 30 consituents


The data of the single stock that we will be using for this case study is obtained from Yahoo Finance API. The data contains Open-High-Low-Close price and volume.


<a id='1'></a>
# Part 2. Getting Started- Load Python Packages

<a id='1.1'></a>
## 2.1. Install all the packages through FinRL library


In [1]:
# # ## install finrl library
# !pip install wrds
# !pip install swig
# !pip install -q condacolab
# import condacolab
# condacolab.install()
# !apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git



<a id='1.2'></a>
## 2.2. Check if the additional packages needed are present, if not install them.
* Yahoo Finance API
* pandas
* numpy
* matplotlib
* stockstats
* OpenAI gym
* stable-baselines
* tensorflow
* pyfolio

<a id='1.3'></a>
## 2.3. Import Packages

In [21]:
import warnings
warnings.filterwarnings("ignore")

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import time

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLEnsembleAgent
from finrl.plot import backtest_stats, get_baseline
from finrl.main import check_and_make_directories
from finrl.config_tickers import DOW_30_TICKER
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)
from stable_baselines3.common.vec_env import DummyVecEnv

import sys
sys.path.append("../FinRL-Library")

%matplotlib inline

In [34]:
# Bayesian Ensemble Agent Implementation
from finrl.agents.stablebaselines3.models import MODELS

# These are needed by the static methods replicated from DRLEnsembleAgent
from finrl.agents.stablebaselines3.models import (
    MODEL_KWARGS,
    NOISE,
    TensorboardCallback,
)
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from finrl import config
import statistics


class FixedTensorboardCallback(BaseCallback):
    """Fixed TensorboardCallback that handles both on-policy (rollout_buffer)
    and off-policy (replay_buffer) algorithms."""

    def __init__(self, verbose=0):
        super().__init__(verbose)

    def _on_step(self) -> bool:
        try:
            self.logger.record(key="train/reward", value=self.locals["rewards"][0])
        except BaseException:
            try:
                self.logger.record(key="train/reward", value=self.locals["reward"][0])
            except BaseException:
                self.logger.record(key="train/reward", value=None)
        return True

    def _on_rollout_end(self) -> bool:
        try:
            # On-policy algorithms (A2C, PPO) use rollout_buffer
            if "rollout_buffer" in self.locals:
                rewards = self.locals["rollout_buffer"].rewards.flatten()
            # Off-policy algorithms (DDPG, SAC, TD3) use replay_buffer
            elif "replay_buffer" in self.locals:
                replay_buf = self.locals["replay_buffer"]
                if replay_buf.full:
                    rewards = replay_buf.rewards[:replay_buf.buffer_size].flatten()
                else:
                    rewards = replay_buf.rewards[:replay_buf.pos].flatten()
            else:
                rewards = None

            if rewards is not None and len(rewards) > 0:
                self.logger.record(key="train/reward_min", value=min(rewards))
                self.logger.record(key="train/reward_mean", value=statistics.mean(rewards))
                self.logger.record(key="train/reward_max", value=max(rewards))
            else:
                self.logger.record(key="train/reward_min", value=None)
                self.logger.record(key="train/reward_mean", value=None)
                self.logger.record(key="train/reward_max", value=None)
        except BaseException:
            self.logger.record(key="train/reward_min", value=None)
            self.logger.record(key="train/reward_mean", value=None)
            self.logger.record(key="train/reward_max", value=None)
        return True


class DRLEnsembleAgentBayesian:
    """Bayesian Optimization-based DRL Ensemble Agent.

    Implements dynamic model weighting using the methodology from the paper:
    - Cold-start phase (i=1, i_M = i_c in N_m): weight = 1
    - Adaptive phase (i>i_c): w_{i_M}(t) = l * w_{i_M}(t-1) + (1-l) * w_hat_{i_M}(t)
    - w_hat_{i_M}(t) computed via inverse accumulated error with discount factor gamma
    - ranking_metric selects the weighting scheme: IMSE, softmax, or IMAE
    """

    # ------------------------------------------------------------------ #
    # Static helpers (copied verbatim from DRLEnsembleAgent)
    # ------------------------------------------------------------------ #

    @staticmethod
    def get_model(
        model_name,
        env,
        policy="MlpPolicy",
        policy_kwargs=None,
        model_kwargs=None,
        seed=None,
        verbose=1,
    ):
        if model_name not in MODELS:
            raise ValueError(f"Model '{model_name}' not found in MODELS.")

        if model_kwargs is None:
            temp_model_kwargs = MODEL_KWARGS[model_name]
        else:
            temp_model_kwargs = model_kwargs.copy()

        if "action_noise" in temp_model_kwargs:
            n_actions = env.action_space.shape[-1]
            temp_model_kwargs["action_noise"] = NOISE[
                temp_model_kwargs["action_noise"]
            ](mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))
        print(temp_model_kwargs)
        return MODELS[model_name](
            policy=policy,
            env=env,
            tensorboard_log=f"{config.TENSORBOARD_LOG_DIR}/{model_name}",
            verbose=verbose,
            policy_kwargs=policy_kwargs,
            seed=seed,
            **temp_model_kwargs,
        )

    @staticmethod
    def train_model(
        model,
        model_name,
        tb_log_name,
        iter_num,
        total_timesteps=5000,
        callbacks=None,
    ):
        model = model.learn(
            total_timesteps=total_timesteps,
            tb_log_name=tb_log_name,
            callback=(
                CallbackList(
                    [FixedTensorboardCallback()]
                    + [callback for callback in callbacks]
                )
                if callbacks is not None
                else FixedTensorboardCallback()
            ),
        )
        model.save(
            f"{config.TRAINED_MODEL_DIR}/{model_name.upper()}_{total_timesteps // 1000}k_{iter_num}"
        )
        return model

    @staticmethod
    def get_validation_sharpe(iteration, model_name):
        """Calculate Sharpe ratio based on validation results."""
        df_total_value = pd.read_csv(
            f"results/account_value_validation_{model_name}_{iteration}.csv"
        )
        if df_total_value["daily_return"].var() == 0:
            if df_total_value["daily_return"].mean() > 0:
                return np.inf
            else:
                return 0.0
        else:
            return (
                (4 ** 0.5)
                * df_total_value["daily_return"].mean()
                / df_total_value["daily_return"].std()
            )

    # ------------------------------------------------------------------ #
    # Constructor
    # ------------------------------------------------------------------ #

    def __init__(
        self,
        df,
        train_period,
        val_test_period,
        rebalance_window,
        validation_window,
        stock_dim,
        hmax,
        initial_amount,
        buy_cost_pct,
        sell_cost_pct,
        reward_scaling,
        state_space,
        action_space,
        tech_indicator_list,
        print_verbosity,
        bayesian_params=None,
    ):
        self.df = df
        self.train_period = train_period
        self.val_test_period = val_test_period

        self.unique_trade_date = df[
            (df.date > val_test_period[0]) & (df.date <= val_test_period[1])
        ].date.unique()
        self.rebalance_window = rebalance_window
        self.validation_window = validation_window

        self.stock_dim = stock_dim
        self.hmax = hmax
        self.initial_amount = initial_amount
        self.buy_cost_pct = buy_cost_pct
        self.sell_cost_pct = sell_cost_pct
        self.reward_scaling = reward_scaling
        self.state_space = state_space
        self.action_space = action_space
        self.tech_indicator_list = tech_indicator_list
        self.print_verbosity = print_verbosity
        self.train_env = None

        # Bayesian optimisation parameters
        self.bayesian_params = bayesian_params or {}
        self.i_c = self.bayesian_params.get("i_c", 1)       # Cold-start model index
        self.n_m = self.bayesian_params.get("n_m", 5)       # Max models to combine
        self.e_w = self.bayesian_params.get("e_w", 63)      # Evaluation window
        self.gamma = self.bayesian_params.get("phi", 0.95)   # Discount factor (gamma/phi)
        self.l = self.bayesian_params.get("l", 0.5)          # Update parameter (lambda)
        self.ranking_metric = self.bayesian_params.get("ranking_metric", "mse")

        # State carried across rebalance iterations
        self.model_weights = {}       # w_{i_M}(t-1)
        self.iteration_count = 0

    # ------------------------------------------------------------------ #
    # Instance methods (copied verbatim from DRLEnsembleAgent)
    # ------------------------------------------------------------------ #

    def DRL_validation(self, model, test_data, test_env, test_obs):
        """Run model through validation environment."""
        for _ in range(len(test_data.index.unique())):
            action, _states = model.predict(test_obs)
            test_obs, rewards, dones, info = test_env.step(action)

    def DRL_prediction(
        self, model, name, last_state, iter_num, turbulence_threshold, initial
    ):
        """Make a prediction based on trained model."""
        trade_data = data_split(
            self.df,
            start=self.unique_trade_date[iter_num - self.rebalance_window],
            end=self.unique_trade_date[iter_num],
        )
        trade_env = DummyVecEnv(
            [
                lambda: StockTradingEnv(
                    df=trade_data,
                    stock_dim=self.stock_dim,
                    hmax=self.hmax,
                    initial_amount=self.initial_amount,
                    num_stock_shares=[0] * self.stock_dim,
                    buy_cost_pct=[self.buy_cost_pct] * self.stock_dim,
                    sell_cost_pct=[self.sell_cost_pct] * self.stock_dim,
                    reward_scaling=self.reward_scaling,
                    state_space=self.state_space,
                    action_space=self.action_space,
                    tech_indicator_list=self.tech_indicator_list,
                    turbulence_threshold=turbulence_threshold,
                    initial=initial,
                    previous_state=last_state,
                    model_name=name,
                    mode="trade",
                    iteration=iter_num,
                    print_verbosity=self.print_verbosity,
                )
            ]
        )

        trade_obs = trade_env.reset()
        for i in range(len(trade_data.index.unique())):
            action, _states = model.predict(trade_obs)
            trade_obs, rewards, dones, info = trade_env.step(action)
            if i == (len(trade_data.index.unique()) - 2):
                last_state = trade_env.envs[0].render()

        df_last_state = pd.DataFrame({"last_state": last_state})
        df_last_state.to_csv(f"results/last_state_{name}_{i}.csv", index=False)
        return last_state

    def _train_window(
        self,
        model_name,
        model_kwargs,
        sharpe_list,
        validation_start_date,
        validation_end_date,
        timesteps_dict,
        i,
        validation,
        turbulence_threshold,
    ):
        """Train the model for a single window."""
        if model_kwargs is None:
            return None, sharpe_list, -1

        print(f"======{model_name} Training========")
        model = self.get_model(
            model_name, self.train_env, policy="MlpPolicy", model_kwargs=model_kwargs
        )
        model = self.train_model(
            model,
            model_name,
            tb_log_name=f"{model_name}_{i}",
            iter_num=i,
            total_timesteps=timesteps_dict[model_name],
        )
        print(
            f"======{model_name} Validation from: ",
            validation_start_date,
            "to ",
            validation_end_date,
        )
        val_env = DummyVecEnv(
            [
                lambda: StockTradingEnv(
                    df=validation,
                    stock_dim=self.stock_dim,
                    hmax=self.hmax,
                    initial_amount=self.initial_amount,
                    num_stock_shares=[0] * self.stock_dim,
                    buy_cost_pct=[self.buy_cost_pct] * self.stock_dim,
                    sell_cost_pct=[self.sell_cost_pct] * self.stock_dim,
                    reward_scaling=self.reward_scaling,
                    state_space=self.state_space,
                    action_space=self.action_space,
                    tech_indicator_list=self.tech_indicator_list,
                    turbulence_threshold=turbulence_threshold,
                    iteration=i,
                    model_name=model_name,
                    mode="validation",
                    print_verbosity=self.print_verbosity,
                )
            ]
        )
        val_obs = val_env.reset()
        self.DRL_validation(
            model=model,
            test_data=validation,
            test_env=val_env,
            test_obs=val_obs,
        )
        sharpe = self.get_validation_sharpe(i, model_name=model_name)
        print(f"{model_name} Sharpe Ratio: ", sharpe)
        sharpe_list.append(sharpe)
        return model, sharpe_list, sharpe

    # ------------------------------------------------------------------ #
    # Bayesian weighting helpers
    # ------------------------------------------------------------------ #

    def _calculate_model_errors(self, model_dct, ranking_metric="mse"):
        """Compute per-model error/score used for ranking.

        Parameters
        ----------
        model_dct : dict
            Keys are model names; each value dict must contain 'sharpe'.
        ranking_metric : str
            'mse' (inverse mean squared error), 'softmax', or
            'imae' (inverse mean absolute error).

        Returns
        -------
        dict
            {model_name: error} -- lower is better.
        """
        errors = {}
        sharpes = {k: v.get("sharpe", -1) for k, v in model_dct.items()}

        if ranking_metric == "mse":
            for model_name, sharpe in sharpes.items():
                if sharpe > 0:
                    errors[model_name] = 1.0 / (1.0 + sharpe)
                else:
                    errors[model_name] = 1.0
        elif ranking_metric == "softmax":
            sharpe_array = np.array(list(sharpes.values()))
            sharpe_array = np.maximum(sharpe_array, 0)
            softmax_weights = np.exp(sharpe_array) / np.sum(np.exp(sharpe_array))
            for idx, model_name in enumerate(sharpes.keys()):
                errors[model_name] = 1.0 - softmax_weights[idx]
        elif ranking_metric == "imae":
            for model_name, sharpe in sharpes.items():
                if sharpe > 0:
                    errors[model_name] = 1.0 / (1.0 + np.abs(sharpe))
                else:
                    errors[model_name] = 1.0
        return errors

    def _calculate_dynamic_weights(self, model_dct, iteration, ranking_metric="mse"):
        """Calculate dynamic weights for the ensemble (Bayesian approach).

        Implements the weighting scheme from the paper:
        - Select top n_m models by accumulated (discounted) error -> N_m(t)
        - Compute raw weight w_hat from inverse error
        - Cold-start (iteration <= i_c): w = 1 for best model
        - Adaptive (iteration > i_c):
              w_{i_M}(t) = l * w_{i_M}(t-1) + (1-l) * w_hat_{i_M}(t)
        """
        errors = self._calculate_model_errors(model_dct, ranking_metric)

        # Rank models by error (ascending -- lower error is better)
        ranked_models = sorted(errors.items(), key=lambda x: x[1])
        selected_models = ranked_models[: min(self.n_m, len(ranked_models))]
        selected_names = {name for name, _ in selected_models}

        weights = {}

        if iteration <= self.i_c:
            # Cold-start: assign weight = 1 to the single best model (i_c)
            best_model = ranked_models[0][0]
            for model_name in errors:
                weights[model_name] = 1.0 if model_name == best_model else 0.0
        else:
            # Compute raw w_hat from inverse error for selected models
            total_inv_err = sum(1.0 / (e + 1e-8) for _, e in selected_models)
            for model_name in errors:
                if model_name in selected_names:
                    err = errors[model_name]
                    w_hat = (1.0 / (err + 1e-8)) / total_inv_err
                    if model_name in self.model_weights:
                        # Paper: w(t) = l * w(t-1) + (1-l) * w_hat(t)
                        weights[model_name] = (
                            self.l * self.model_weights[model_name]
                            + (1 - self.l) * w_hat
                        )
                    else:
                        weights[model_name] = w_hat
                else:
                    weights[model_name] = 0.0

            # Normalise so weights sum to 1
            total = sum(weights.values())
            if total > 0:
                weights = {k: v / total for k, v in weights.items()}

        # Store for next iteration
        self.model_weights = weights.copy()
        return weights

    # ------------------------------------------------------------------ #
    # Main ensemble loop
    # ------------------------------------------------------------------ #

    def run_ensemble_strategy(
        self,
        A2C_model_kwargs,
        PPO_model_kwargs,
        DDPG_model_kwargs,
        SAC_model_kwargs,
        TD3_model_kwargs,
        timesteps_dict,
    ):
        """Run Bayesian ensemble strategy with dynamic model weighting.

        Returns
        -------
        df_summary : pd.DataFrame
            Per-iteration Sharpe ratios and selected model.
        model_weights_history : list[dict]
            Weight dict for every iteration.
        """
        kwargs = {
            "a2c": A2C_model_kwargs,
            "ppo": PPO_model_kwargs,
            "ddpg": DDPG_model_kwargs,
            "sac": SAC_model_kwargs,
            "td3": TD3_model_kwargs,
        }

        model_dct = {k: {"sharpe_list": [], "sharpe": -1} for k in MODELS.keys()}

        print("============Start Bayesian Ensemble Strategy============")

        last_state_ensemble = []
        model_use = []
        model_weights_list = []
        validation_start_date_list = []
        validation_end_date_list = []
        iteration_list = []

        insample_turbulence = self.df[
            (self.df.date < self.train_period[1])
            & (self.df.date >= self.train_period[0])
        ]
        insample_turbulence_threshold = np.quantile(
            insample_turbulence.turbulence.values, 0.90
        )

        start = time.time()

        for i in range(
            self.rebalance_window + self.validation_window,
            len(self.unique_trade_date),
            self.rebalance_window,
        ):
            self.iteration_count += 1

            validation_start_date = self.unique_trade_date[
                i - self.rebalance_window - self.validation_window
            ]
            validation_end_date = self.unique_trade_date[i - self.rebalance_window]

            validation_start_date_list.append(validation_start_date)
            validation_end_date_list.append(validation_end_date)
            iteration_list.append(i)

            print("============================================")

            if i - self.rebalance_window - self.validation_window == 0:
                initial = True
            else:
                initial = False

            # Turbulence index tuning
            end_date_index = self.df.index[
                self.df["date"]
                == self.unique_trade_date[
                    i - self.rebalance_window - self.validation_window
                ]
            ].to_list()[-1]
            start_date_index = end_date_index - 63 + 1

            historical_turbulence = self.df.iloc[
                start_date_index : (end_date_index + 1), :
            ]
            historical_turbulence = historical_turbulence.drop_duplicates(
                subset=["date"]
            )
            historical_turbulence_mean = np.mean(
                historical_turbulence.turbulence.values
            )

            if historical_turbulence_mean > insample_turbulence_threshold:
                turbulence_threshold = insample_turbulence_threshold
            else:
                turbulence_threshold = np.quantile(
                    insample_turbulence.turbulence.values, 1
                )

            turbulence_threshold = np.quantile(
                insample_turbulence.turbulence.values, 0.99
            )
            print("turbulence_threshold: ", turbulence_threshold)

            # Environment setup
            train = data_split(
                self.df,
                start=self.train_period[0],
                end=self.unique_trade_date[
                    i - self.rebalance_window - self.validation_window
                ],
            )
            self.train_env = DummyVecEnv(
                [
                    lambda: StockTradingEnv(
                        df=train,
                        stock_dim=self.stock_dim,
                        hmax=self.hmax,
                        initial_amount=self.initial_amount,
                        num_stock_shares=[0] * self.stock_dim,
                        buy_cost_pct=[self.buy_cost_pct] * self.stock_dim,
                        sell_cost_pct=[self.sell_cost_pct] * self.stock_dim,
                        reward_scaling=self.reward_scaling,
                        state_space=self.state_space,
                        action_space=self.action_space,
                        tech_indicator_list=self.tech_indicator_list,
                        print_verbosity=self.print_verbosity,
                    )
                ]
            )

            validation = data_split(
                self.df,
                start=self.unique_trade_date[
                    i - self.rebalance_window - self.validation_window
                ],
                end=self.unique_trade_date[i - self.rebalance_window],
            )

            print(
                "======Model training from: ",
                self.train_period[0],
                "to ",
                self.unique_trade_date[
                    i - self.rebalance_window - self.validation_window
                ],
            )

            # Train each model
            for model_name in MODELS.keys():
                model, sharpe_list, sharpe = self._train_window(
                    model_name,
                    kwargs[model_name],
                    model_dct[model_name]["sharpe_list"],
                    validation_start_date,
                    validation_end_date,
                    timesteps_dict,
                    i,
                    validation,
                    turbulence_threshold,
                )
                model_dct[model_name]["sharpe_list"] = sharpe_list
                model_dct[model_name]["model"] = model
                model_dct[model_name]["sharpe"] = sharpe

            # Calculate dynamic weights using Bayesian approach
            weights = self._calculate_dynamic_weights(
                model_dct, self.iteration_count, self.ranking_metric
            )

            weights_str = ", ".join(
                [f"{k}: {v:.4f}" for k, v in weights.items()]
            )
            print(f"Model Weights: {weights_str}")
            model_weights_list.append(weights.copy())

            # Select best model based on highest Sharpe
            sharpes = [model_dct[k]["sharpe"] for k in MODELS.keys()]
            max_mod = list(MODELS.keys())[np.argmax(sharpes)]
            model_use.append(max_mod.upper())

            print(
                "======Trading from: ",
                self.unique_trade_date[i - self.rebalance_window],
                "to ",
                self.unique_trade_date[i],
            )

            last_state_ensemble = self.DRL_prediction(
                model=model_dct[max_mod]["model"],
                name="ensemble_bayesian",
                last_state=last_state_ensemble,
                iter_num=i,
                turbulence_threshold=turbulence_threshold,
                initial=initial,
            )

        end = time.time()
        print(f"Bayesian Ensemble Strategy took: {(end - start) / 60:.2f} minutes")

        df_summary = pd.DataFrame(
            [
                iteration_list,
                validation_start_date_list,
                validation_end_date_list,
                model_use,
                model_dct["a2c"]["sharpe_list"],
                model_dct["ppo"]["sharpe_list"],
                model_dct["ddpg"]["sharpe_list"],
                model_dct["sac"]["sharpe_list"],
                model_dct["td3"]["sharpe_list"],
            ]
        ).T
        df_summary.columns = [
            "Iter",
            "Val Start",
            "Val End",
            "Model Used",
            "A2C Sharpe",
            "PPO Sharpe",
            "DDPG Sharpe",
            "SAC Sharpe",
            "TD3 Sharpe",
        ]

        return df_summary, model_weights_list

<a id='1.4'></a>
## 2.4. Create Folders

In [35]:
check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])

<a id='2'></a>
# Part 3. Download Data
Yahoo Finance is a website that provides stock data, financial news, financial reports, etc. All the data provided by Yahoo Finance is free.
* FinRL uses a class **YahooDownloader** to fetch data from Yahoo Finance API
* Call Limit: Using the Public API (without authentication), you are limited to 2,000 requests per hour per IP (or up to a total of 48,000 requests a day).




-----
class YahooDownloader:
    Provides methods for retrieving daily stock data from
    Yahoo Finance API

    Attributes
    ----------
        start_date : str
            start date of the data (modified from config.py)
        end_date : str
            end date of the data (modified from config.py)
        ticker_list : list
            a list of stock tickers (modified from config.py)

    Methods
    -------
    fetch_data()
        Fetches data from yahoo API


In [36]:
print(DOW_30_TICKER)

['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WBA', 'WMT', 'DIS', 'DOW']


In [26]:
df = YahooDownloader(start_date = TRAIN_START_DATE,
                     end_date = TEST_END_DATE,
                     ticker_list = DOW_30_TICKER).fetch_data()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (55212, 8)


# Part 4: Preprocess Data
Data preprocessing is a crucial step for training a high quality machine learning model. We need to check for missing data and do feature engineering in order to convert the data into a model-ready state.
* Add technical indicators. In practical trading, various information needs to be taken into account, for example the historical stock prices, current holding shares, technical indicators, etc. In this article, we demonstrate two trend-following technical indicators: MACD and RSI.
* Add turbulence index. Risk-aversion reflects whether an investor will choose to preserve the capital. It also influences one's trading strategy when facing different market volatility level. To control the risk in a worst-case scenario, such as financial crisis of 2007–2008, FinRL employs the financial turbulence index that measures extreme asset price fluctuation.

In [27]:
fe = FeatureEngineer(use_technical_indicator=True,
                     tech_indicator_list = INDICATORS,
                     use_turbulence=True,
                     user_defined_feature = False)

processed = fe.preprocess_data(df)
processed = processed.copy()
processed = processed.fillna(0)
processed = processed.replace(np.inf,0)

Successfully added technical indicators
Successfully added turbulence index


<a id='4'></a>
# Part 5. Design Environment
Considering the stochastic and interactive nature of the automated stock trading tasks, a financial task is modeled as a **Markov Decision Process (MDP)** problem. The training process involves observing stock price change, taking an action and reward's calculation to have the agent adjusting its strategy accordingly. By interacting with the environment, the trading agent will derive a trading strategy with the maximized rewards as time proceeds.

Our trading environments, based on OpenAI Gym framework, simulate live stock markets with real market data according to the principle of time-driven simulation.

The action space describes the allowed actions that the agent interacts with the environment. Normally, action a includes three actions: {-1, 0, 1}, where -1, 0, 1 represent selling, holding, and buying one share. Also, an action can be carried upon multiple shares. We use an action space {-k,…,-1, 0, 1, …, k}, where k denotes the number of shares to buy and -k denotes the number of shares to sell. For example, "Buy 10 shares of AAPL" or "Sell 10 shares of AAPL" are 10 or -10, respectively. The continuous action space needs to be normalized to [-1, 1], since the policy is defined on a Gaussian distribution, which needs to be normalized and symmetric.

In [37]:
stock_dimension = len(processed.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 28, State Space: 281


In [38]:
env_kwargs_dynamic = {
    "hmax": 100,
    "initial_amount": 1000000,
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4,
    "print_verbosity":5
}

# Part 6: Bayesian Optimization-based Ensemble Strategy

## Methodology Overview

The **DRLEnsembleAgentBayesian** class implements a systematic Bayesian optimization approach for combining multiple DRL algorithms. This methodology improves upon the basic ensemble approach by:

1. **Dynamic Weight Calculation**: Instead of simply selecting the best model, we assign adaptive weights to multiple models based on their historical performance.

2. **Cold-Start Phase**: During initial iterations (i ≤ i_c), the ensemble uses only the top-performing models, concentrating on proven strategies.

3. **Adaptive Phase**: After the cold-start phase, weights are updated using historical information and current performance, incorporating both past success and new evidence.

## Key Parameters

- **i_c (Cold-start index)**: Number of iterations to use only top models. Example: i_c=1 means cold-start for first iteration only.
- **n_m (Maximum models)**: Maximum number of models to include in the weighted ensemble. Example: n_m=3 limits ensemble to top 3 performers.
- **e_w (Evaluation window)**: Number of observations used to calculate weights (in days).
- **φ (Discount factor)**: Emphasizes recent observations (0 < φ ≤ 1). Higher values give more weight to recent performance.
- **λ (Update parameter)**: Balances historical vs. current weights. Example: λ=0.5 means 50% historical, 50% current.
- **ranking_metric**: How to rank models:
  - 'mse': Mean squared error (inverse Sharpe ratio)
  - 'softmax': Softmax normalization of Sharpe ratios
  - 'imae': Inverse mean absolute error

## Weight Calculation Formula

For the adaptive phase (t > i_c):

$$w_{i,M}(t) = (1 - \lambda) \cdot w_{i,M}(t-1) + \lambda \cdot w'_{i,M}(t)$$

Where:
- $w_{i,M}(t)$ is the weight of model i at time t
- $w_{i,M}(t-1)$ is the historical weight
- $w'_{i,M}(t)$ is the current weight based on validation performance
- $\lambda$ controls the balance between stability and adaptability

* The implementation of the DRL algorithms are based on **OpenAI Baselines** and **Stable Baselines**. Stable Baselines is a fork of OpenAI Baselines, with a major structural refactoring, and code cleanups.
* FinRL library includes fine-tuned standard DRL algorithms, such as DQN, DDPG,
Multi-Agent DDPG, PPO, SAC, A2C and TD3. We also allow users to
design their own DRL algorithms by adapting these DRL algorithms.

* In this notebook, we are training and validating 3 agents (A2C, PPO, DDPG) using Rolling-window Ensemble Method ([reference code](https://github.com/AI4Finance-LLC/Deep-Reinforcement-Learning-for-Automated-Stock-Trading-Ensemble-Strategy-ICAIF-2020/blob/80415db8fa7b2179df6bd7e81ce4fe8dbf913806/model/models.py#L92))

In [39]:
# Configure Bayesian ensemble parameters
bayesian_params = {
    'i_c': 1,              # Cold-start index: use only top model(s) initially
    'n_m': 3,              # Maximum number of models to combine
    'e_w': 63,             # Evaluation window for weight calculation
    'phi': 0.95,           # Discount factor for emphasizing recent observations
    'l': 0.5,              # Update parameter balancing historical vs current weights
    'ranking_metric': 'mse'  # Options: 'mse', 'softmax', 'imae'
}

rebalance_window = 63 # rebalance_window is the number of days to retrain the model
validation_window = 63 # validation_window is the number of days to do validation and trading (e.g. if validation_window=63, then both validation and trading period will be 63 days)

# Create Bayesian ensemble agent
ensemble_agent_bayesian = DRLEnsembleAgentBayesian(
    df=processed,
    train_period=(TRAIN_START_DATE, TRAIN_END_DATE),
    val_test_period=(TEST_START_DATE, TEST_END_DATE),
    rebalance_window=rebalance_window,
    validation_window=validation_window,
    bayesian_params=bayesian_params,
    **env_kwargs_dynamic
)

print("Bayesian Ensemble Agent configured with parameters:")
print(f"  Cold-start index (i_c): {bayesian_params['i_c']}")
print(f"  Max models (n_m): {bayesian_params['n_m']}")
print(f"  Evaluation window (e_w): {bayesian_params['e_w']}")
print(f"  Discount factor (φ): {bayesian_params['phi']}")
print(f"  Update parameter (λ): {bayesian_params['l']}")
print(f"  Ranking metric: {bayesian_params['ranking_metric']}")

Bayesian Ensemble Agent configured with parameters:
  Cold-start index (i_c): 1
  Max models (n_m): 3
  Evaluation window (e_w): 63
  Discount factor (φ): 0.95
  Update parameter (λ): 0.5
  Ranking metric: mse


In [40]:
A2C_model_kwargs = {
                    'n_steps': 5,
                    'ent_coef': 0.005,
                    'learning_rate': 0.0007
                    }

PPO_model_kwargs = {
                    "ent_coef":0.01,
                    "n_steps": 2048,
                    "learning_rate": 0.00025,
                    "batch_size": 128
                    }

DDPG_model_kwargs = {
                      #"action_noise":"ornstein_uhlenbeck",
                      "buffer_size": 10_000,
                      "learning_rate": 0.0005,
                      "batch_size": 64
                    }

SAC_model_kwargs = {
    "batch_size": 64,
    "buffer_size": 100000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1",
}

TD3_model_kwargs = {"batch_size": 100, "buffer_size": 1000000, "learning_rate": 0.0001}

timesteps_dict = {'a2c' : 10_000,
                 'ppo' : 10_000,
                 'ddpg' : 10_000,
                 'sac' : 10_000,
                 'td3' : 10_000
                 }

In [ ]:
# Run Bayesian Ensemble Strategy
df_summary, model_weights_history = ensemble_agent_bayesian.run_ensemble_strategy(
    A2C_model_kwargs,
    PPO_model_kwargs,
    DDPG_model_kwargs,
    SAC_model_kwargs,
    TD3_model_kwargs,
    timesteps_dict
)

============Start Bayesian Ensemble Strategy============
turbulence_threshold:  239.37673545691695
======Model training from:  2014-01-06 to  2020-08-03
======a2c Training========
{'n_steps': 5, 'ent_coef': 0.005, 'learning_rate': 0.0007}
Using cpu device
Logging to tensorboard_log/a2c\a2c_126_2
---------------------------------------
| time/                 |             |
|    fps                | 180         |
|    iterations         | 100         |
|    time_elapsed       | 2           |
|    total_timesteps    | 500         |
| train/                |             |
|    entropy_loss       | -39.9       |
|    explained_variance | -0.0365     |
|    learning_rate      | 0.0007      |
|    n_updates          | 99          |
|    policy_loss        | -17.8       |
|    reward             | -0.71470004 |
|    reward_max         | 1.3991853   |
|    reward_mean        | 0.27945143  |
|    reward_min         | -0.71470004 |
|    std                | 1.01        |
|    value_loss        

In [ ]:
# Display Bayesian Ensemble Summary
print("=== Bayesian Ensemble Strategy Results ===")
print(df_summary)
print("\n=== Model Weights Evolution ===")
for iteration, weights in enumerate(model_weights_history):
    print(f"Iteration {iteration + 1}:")
    for model_name, weight in weights.items():
        print(f"  {model_name.upper()}: {weight:.4f}")
    print()

In [ ]:
# Visualize Model Weights Evolution Over Time

# Prepare data for plotting
if model_weights_history:
    models = list(model_weights_history[0].keys())
    iterations = list(range(1, len(model_weights_history) + 1))
    
    # Extract weights for each model
    weights_by_model = {model: [] for model in models}
    for weights_dict in model_weights_history:
        for model, weight in weights_dict.items():
            weights_by_model[model].append(weight)
    
    # Create figure
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Plot 1: Stacked area chart of all model weights
    ax1 = axes[0]
    bottom = np.zeros(len(iterations))
    colors = plt.cm.Set3(np.linspace(0, 1, len(models)))
    
    for idx, model in enumerate(models):
        ax1.fill_between(iterations, bottom, bottom + np.array(weights_by_model[model]), 
                         label=model.upper(), alpha=0.8, cxolor=colors[idx])
        bottom += np.array(weights_by_model[model])
    
    ax1.set_xlabel('Iteration', fontsize=12)
    ax1.set_ylabel('Model Weight', fontsize=12)
    ax1.set_title('Dynamic Model Weights Evolution (Stacked Area)', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper left', ncol=3)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 1])
    
    # Plot 2: Line plot for each model's weight
    ax2 = axes[1]
    for idx, model in enumerate(models):
        ax2.plot(iterations, weights_by_model[model], marker='o', label=model.upper(), 
                linewidth=2, markersize=5, color=colors[idx])
    
    ax2.set_xlabel('Iteration', fontsize=12)
    ax2.set_ylabel('Model Weight', fontsize=12)
    ax2.set_title('Individual Model Weights Over Time', fontsize=14, fontweight='bold')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig('model_weights_evolution.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("Model weights visualization saved as 'model_weights_evolution.png'")

<a id='6'></a>
# Part 7: Backtest Our Strategy
Backtesting plays a key role in evaluating the performance of a trading strategy. Automated backtesting tool is preferred because it reduces the human error. We usually use the Quantopian pyfolio package to backtest our trading strategies. It is easy to use and consists of various individual plots that provide a comprehensive image of the performance of a trading strategy.

In [ ]:
unique_trade_date = processed[(processed.date > TEST_START_DATE)&(processed.date <= TEST_END_DATE)].date.unique()

In [ ]:
df_trade_date = pd.DataFrame({'datadate':unique_trade_date})

df_account_value=pd.DataFrame()
for i in range(rebalance_window+validation_window, len(unique_trade_date)+1,rebalance_window):
    temp = pd.read_csv('results/account_value_trade_{}_{}.csv'.format('ensemble',i))
    df_account_value = df_account_value.append(temp,ignore_index=True)
sharpe=(252**0.5)*df_account_value.account_value.pct_change(1).mean()/df_account_value.account_value.pct_change(1).std()
print('Sharpe Ratio: ',sharpe)
df_account_value=df_account_value.join(df_trade_date[validation_window:].reset_index(drop=True))

In [ ]:
df_account_value.head()

In [ ]:
%matplotlib inline
df_account_value.account_value.plot()

<a id='6.1'></a>
## 7.1 BackTestStats
pass in df_account_value, this information is stored in env class


In [ ]:
print("==============Get Backtest Results===========")
now = datetime.datetime.now().strftime('%Y%m%d-%Hh%M')

perf_stats_all = backtest_stats(account_value=df_account_value)
perf_stats_all = pd.DataFrame(perf_stats_all)

In [ ]:
#baseline stats
print("==============Get Baseline Stats===========")
df_dji_ = get_baseline(
        ticker="^DJI",
        start = df_account_value.loc[0,'date'],
        end = df_account_value.loc[len(df_account_value)-1,'date'])

stats = backtest_stats(df_dji_, value_col_name = 'close')

In [ ]:
df_dji = pd.DataFrame()
df_dji['date'] = df_account_value['date']
df_dji['dji'] = df_dji_['close'] / df_dji_['close'][0] * env_kwargs_dynamic["initial_amount"]
print("df_dji: ", df_dji)
df_dji.to_csv("df_dji.csv")
df_dji = df_dji.set_index(df_dji.columns[0])
print("df_dji: ", df_dji)
df_dji.to_csv("df_dji+.csv")

df_account_value.to_csv('df_account_value.csv')


<a id='6.2'></a>
## 7.2 BackTestPlot

In [ ]:

# print("==============Compare to DJIA===========")
# %matplotlib inline
# # S&P 500: ^GSPC
# # Dow Jones Index: ^DJI
# # NASDAQ 100: ^NDX
# backtest_plot(df_account_value,
#               baseline_ticker = '^DJI',
#               baseline_start = df_account_value.loc[0,'date'],
#               baseline_end = df_account_value.loc[len(df_account_value)-1,'date'])
df.to_csv("df.csv")
df_result_ensemble = pd.DataFrame({'date': df_account_value['date'], 'ensemble':df_account_value['account_value']})
df_result_ensemble = df_result_ensemble.set_index('date')

print("df_result_ensemble.columns: ", df_result_ensemble.columns)

print("df_trade_date: ", df_trade_date)
# df_result_ensemble['date'] = df_trade_date['datadate']
# df_result_ensemble['account_value'] = df_account_value['account_value']
df_result_ensemble.to_csv("df_result_ensemble.csv")
print("df_result_ensemble: ", df_result_ensemble)
print("==============Compare to DJIA===========")
result = pd.DataFrame()
# result = pd.merge(result, df_result_ensemble, left_index=True, right_index=True)

result = pd.merge(df_result_ensemble, df_dji, left_index=True, right_index=True)
print("result: ", result)
result.to_csv("result.csv")
result.columns = ['ensemble', 'dji']

%matplotlib inline
plt.rcParams["figure.figsize"] = (15, 5)
plt.figure()
result.plot()